In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [2]:
%%writefile src/model.py
import numpy as np
from src import metrics

def log5(pa,pb):
    return (pa-pa*pb)/(pa+pb-2*pa*pb)
def game_prob(pa,pb,a_is_home,hfa=0.023):
    p=log5(pa,pb)+(hfa if a_is_home else -hfa)
    return min(max(p,0.05),0.95)


Writing src/model.py


In [4]:
from src import model

print(model.log5(0.5,0.5))
print(model.log5(0.65,0.5))

0.5
0.6500000000000001


In [22]:
%%writefile -a src/model.py


def simulate_series(pa,pb,home_pattern,n=10000,hfa=0.023,seed=2026,need=4):
    rng=np.random.default_rng(seed)
    prob=[game_prob(pa,pb,h,hfa) for h in home_pattern]
    a_title=0
    results={}
    for _ in range(n):
        wa=wb=0
        for p in prob:
            if rng.random()< p:
                wa+=1
            else:
                wb+=1
            if wa==need or wb==need:
                break
        a_title+=1 if wa==need else 0
        key=f'{wa}-{wb}'
        results[key]=results.get(key,0)+1
    return a_title/n, results, prob


Appending to src/model.py


In [4]:
import pandas as pd
from src import model,config

team=pd.read_csv('data/clean/teams.csv').set_index('team')

p , results,prob=model.simulate_series(team.loc['味全龍','pythag'],team.loc['中信兄弟','pythag'],config.TS_HOME_PATTERN)
print(p)
print(results)

0.6763
{'4-1': 1918, '2-4': 1015, '1-4': 737, '4-0': 1095, '4-2': 2075, '4-3': 1675, '3-4': 1132, '0-4': 353}


In [6]:
n=[100,1000,100000]

import pandas as pd
from src import model,config

team=pd.read_csv('data/clean/teams.csv').set_index('team')
for time in n:
    p , _,_=model.simulate_series(team.loc['味全龍','pythag'],team.loc['中信兄弟','pythag'],config.TS_HOME_PATTERN,time,hfa=0)
    print(p)

0.62
0.632
0.65319


In [7]:
!git add .
!git commit -m "Day 11: 系列賽模型"

[main ca46e92] Day 11: 系列賽模型
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 6 files changed, 594 insertions(+), 8 deletions(-)
 create mode 100644 .virtual_documents/D03.ipynb
 create mode 100644 .virtual_documents/D04.ipynb
 create mode 100644 D07.ipynb
 create mode 100644 src/model.py


In [14]:
%%writefile -a src/model.py
def league_context(teams,pitchers,lg_woba):
    ip=pitchers['IP'].sum()
    bip=pitchers['BF'].sum()-pitchers['SO'].sum()-pitchers['BB'].sum()-pitchers['HR'].sum()
    return {
        'R_per_G':teams['RS'].sum()/teams['G'].sum(),
        'wOBA':lg_woba,
        'FIP':(pitchers['FIP_calc']*pitchers['IP']).sum()/ip,
        'DER':1-(pitchers['H']-pitchers['HR']).sum()/bip
    }

Appending to src/model.py


In [15]:
%%writefile -a src/model.py

def team_pitching(pitchers):
    rows=[]
    for abbr, g in pitchers.groupby('abbr'):
        pen=g[g['role']=='後援']
        bip=g['BF'].sum()-g['SO'].sum()-g['BB'].sum()-g['HR'].sum()
        rows.append({
            'abbr':abbr,
            'bullpen_fip':(pen['FIP_calc']*pen['IP']).sum()/pen['IP'].sum(),
            'DER':1-(g['H'].sum()-g['HR'].sum())/bip
        })
    return rows

Appending to src/model.py


In [16]:
%%writefile -a src/model.py
def expected_runs(woba_vs_sp, woba_all, sp_fip, bullpen_fip, opp_der, ctx,
                  sp_innings=6, home=False):
    sp_part=sp_innings/9*(woba_vs_sp/ctx['wOBA'])**2*(sp_fip/ctx['FIP'])
    bp_part=(9-sp_innings)/9*(woba_all/ctx['wOBA'])**2*(bullpen_fip/ctx['FIP'])
    defense=1-(opp_der-ctx['DER'])*4
    park=1.023 if home else 0.977
    return ctx["R_per_G"] * (sp_part + bp_part) * defense * park
def win_prob_from_runs(ra, rb, exp=1.83):
    return metrics.pythag(ra, rb, exp)

Appending to src/model.py


In [3]:
import pandas as pd
from src import metrics, model

team=pd.read_csv('data/clean/teams.csv')
pitchers=pd.read_csv('data/clean/pitchers.csv')
ctx=model.league_context(team,pitchers,lg_woba=0.319)
tp=pd.DataFrame(model.team_pitching(pitchers)).set_index('abbr')
print(ctx)

r=model.expected_runs(0.330,0.319,3.2,tp.loc['龍','bullpen_fip'],tp.loc['龍','DER'],ctx)
print(f'預估得分{r:.2f}')

{'R_per_G': np.float64(3.7222222222222223), 'wOBA': 0.319, 'FIP': np.float64(3.242819843342036), 'DER': np.float64(0.7149668223150651)}
預估得分3.71


In [9]:
from src import model

pen,der=tp.loc['龍','bullpen_fip'],tp.loc['龍','DER']
print([round(model.expected_runs(w,w,3.5,pen,der,ctx),2) for w in [0.300,0.330,0.360]])
print([round(model.expected_runs(0.34,0.34,f,pen,der,ctx),2) for f in[2.5,3.5,4.5]])
home=model.expected_runs(0.34,0.34,2.5,pen,der,ctx,home=True)
away=model.expected_runs(0.34,0.34,2.5,pen,der,ctx,home=False)
print(model.win_prob_from_runs(home,away))

[np.float64(3.33), np.float64(4.03), np.float64(4.8)]
[np.float64(3.46), np.float64(4.28), np.float64(5.1)]
0.521036286793605


In [10]:
!git add .
!git commit -m "Day 12: 單場模型"

[main c4ae5e6] Day 12: 單場模型
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 3 files changed, 275 insertions(+), 1 deletion(-)
